# V6 family distillation — frozen PRO / NANO + BASE / Colab T4

40epochs each,batch8,synthetic=False. Same DWT19 features/scaler/ONNX signature.
PRO is never retrained. **No regenerated partition,no fabricated metrics.**

Before choosing GPU,stage ORIGINAL checkpoint,locked partition and canonical records
beside the two production ONNX/scaler files. Those extra files were NOT found by the
connected Drive audit. Missing inputs stop immediately. This notebook cannot reconstruct
them from model weights or substitute the four-person test fixtures.

The~1CU target is a measured-profile admission rule with an assumed/operator-verified
billing rate,NOT a guarantee. Runtime setup,idle time and retries also cost units.
All tiers are Wacom-trained;touchscreen PD sensitivity is NOT clinically established.

In [ ]:
from pathlib import Path
import time,os,sys,subprocess,threading
JOB_START=time.monotonic()
from google.colab import drive
drive.mount('/content/drive')
PRODUCTION=Path('/content/drive/MyDrive/NEUROTRACE/v6_production')
TEACHER_ONNX=PRODUCTION/'neurotrace_v6_tremorformer.onnx'
SCALER=PRODUCTION/'scaler_config_v6.json'
# Configure actual ORIGINAL files,never replace them with a new split/smoke model.
TEACHER_CHECKPOINT=PRODUCTION/'fold_0_best.pth'
PARTITION=PRODUCTION/'partition.json'
RECORDS=PRODUCTION/'v6_prepared_records.json'
FAMILY=PRODUCTION/'family'
EPOCHS,BATCH_SIZE,TEMPERATURE,ALPHA=40,8,3.,.5
USE_SYNTHETIC=False
CU_PER_HOUR=2.0 # PLANNING ASSUMPTION: verify/update current Colab T4 displayed rate.
CU_ALLOWANCE=.9 # margin below ~1CU; setup since JOB_START counts too.
WALL_SECONDS=3600*CU_ALLOWANCE/CU_PER_HOUR
missing=[str(p) for p in (TEACHER_ONNX,SCALER,TEACHER_CHECKPOINT,PARTITION,RECORDS) if not p.is_file()]
if missing: raise FileNotFoundError('Stage the ORIGINAL private files first; no regenerated split: '+str(missing))

def run_live(command):
    remaining=WALL_SECONDS-(time.monotonic()-JOB_START)
    if remaining<=0: raise RuntimeError('Configured CU/wall-time allowance exhausted')
    print('$',' '.join(map(str,command)),flush=True)
    env=dict(os.environ,PYTHONUNBUFFERED='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
    with subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1,env=env) as process:
        timer=threading.Timer(remaining,process.kill);timer.start()
        try:
            for line in process.stdout: print(line,end='',flush=True)
            code=process.wait()
        finally: timer.cancel()
    if code: raise subprocess.CalledProcessError(code,command)


## Install in fresh supported Python3.10–3.12 runtime

Do not import torch before installation. Select T4; do not apply a T4 billing
assumption to A100. File inputs above remain read-only; only family outputs change.

In [ ]:
REPO=Path('/content/NeuroTrace-AI');BRANCH='arena/4edc99ce-neurotrace-ai'
if not (REPO/'.git').exists():
    run_live(['git','clone','--branch',BRANCH,'https://github.com/VaKhang0709/NeuroTrace-AI.git',REPO])
os.chdir(REPO)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()==BRANCH
run_live(['git','fetch','origin',BRANCH]);run_live(['git','merge','--ff-only','origin/'+BRANCH])
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
if 'torch' in sys.modules: raise RuntimeError('Restart fresh runtime before installing torch')
run_live([sys.executable,'-m','pip','install','-r','requirements-v6-family.txt'])
import torch,json
assert torch.cuda.is_available(),'No CUDA; cannot claim T4 execution'
print(torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0),'Select T4 or explicitly review budget/rate before modifying this guard'
from src.training.family_data import sha256,load_context
ORIGINAL_HASHES={str(p):sha256(p) for p in (TEACHER_ONNX,SCALER,TEACHER_CHECKPOINT,PARTITION,RECORDS)}
records,partition,metadata,context=load_context(RECORDS,PARTITION,SCALER,TEACHER_CHECKPOINT,TEACHER_ONNX)
print('Original partition:',{k:len(v) for k,v in partition.items()})
run_live([sys.executable,'-m','src.models.tremorformer_family'])

## Cached frozen-teacher targets + 40epochs EACH

Uses original teacher-training and teacher-validation identities;never a new draw split.
Binary KL/T² plus BCE,GRL deviceCE. The development-only timing projection must fit
remaining allowance with export reserve;otherwise it aborts,NOT caps data/epochs.
Cached teacher logits are computed only on train parents,never on holdout.

In [ ]:
remaining=int(WALL_SECONDS-(time.monotonic()-JOB_START))
run_live([sys.executable,'-u','-m','src.training.distill',
    '--records',RECORDS,'--partition',PARTITION,'--scaler',SCALER,
    '--teacher-metadata',TEACHER_CHECKPOINT,'--teacher-onnx',TEACHER_ONNX,
    '--output',FAMILY,'--epochs',str(EPOCHS),'--batch-size',str(BATCH_SIZE),
    '--temperature',str(TEMPERATURE),'--alpha',str(ALPHA),'--device','cuda',
    '--max-seconds',str(remaining),'--reserve-seconds','240'])
assert json.loads((FAMILY/'run_status.json').read_text())['complete']

## Dynamic export / size gates / validation-only precision parity

NANO mixed linear INT8≤2MB;BASE FP16≤12MB;PRO copied byte-identically. Inputs/outputs
remain float32/int64,so frontend extraction is unchanged. Fixed tolerances:
FP32<1e-5,FP16<.01,INT8<.10 raw-logit error. Shared scaler is byte-identical;
teacher-only isotonic is NOT applied to student scores. Failed gates stop evaluation.

In [ ]:
run_live([sys.executable,'-u','-m','src.export.export_family',
    '--nano',FAMILY/'nano_best.pth','--base',FAMILY/'base_best.pth',
    '--teacher-checkpoint',TEACHER_CHECKPOINT,'--teacher-onnx',TEACHER_ONNX,
    '--records',RECORDS,'--partition',PARTITION,'--scaler',SCALER,'--output',FAMILY])
manifest=json.loads((FAMILY/'family_manifest.json').read_text())
assert manifest['accepted']
for tier,result in manifest['tiers'].items(): print(tier,result['bytes'],result['sha256'],result['parity'])

## Original15subject holdout — only after acceptance gates

No threshold search,sigmoid/raw0.50,subject-mean aggregation. Report is written ONLY
if every tier and every locked holdout parent succeeds. No fabricated/placeholder
numbers. Single-thread hostCPU512-window latency is a proxy,not measured mobile speed.
Private predictions contain research IDs:do NOT publish them.

In [ ]:
REPORT=Path('reports/private/v6_family_holdout.md')
run_live([sys.executable,'-u','-m','src.eval.benchmark_family',
    '--family',FAMILY,'--teacher-metadata',TEACHER_CHECKPOINT,'--teacher-onnx',TEACHER_ONNX,
    '--records',RECORDS,'--partition',PARTITION,'--output',REPORT])
assert REPORT.is_file()
import shutil
for p in (REPORT,REPORT.with_suffix('.json'),REPORT.with_suffix('.predictions.npz')):
    shutil.copyfile(p,FAMILY/p.name)
assert all(sha256(p)==value for p,value in ORIGINAL_HASHES.items()),'A read-only source artifact changed'
print(REPORT.read_text())
print('Elapsed job seconds:',time.monotonic()-JOB_START)
print('CU estimate at configured rate:',(time.monotonic()-JOB_START)/3600*CU_PER_HOUR)
print('Check actual provider billing; this estimate is not a charge guarantee.')